# LSTM-GATv2 End-to-End (Step 1a)

Per-timestep LSTM-GATv2 with training + architecture fixes.

**Key improvements over `lstm_gat_end_to_end.ipynb`:**
- GATv2 dynamic attention (truly pairwise scores)
- Separate LSTM dropout (0.3) and attention dropout (0.1)
- No LayerNormalization after GAT (matches GCN)
- Larger dimensions: LSTM hidden=32, GAT units=16, heads=4
- Less aggressive gradient clipping: 1.0 vs 0.01
- Batch size 128 for more stable Sharpe estimates

**Architecture**: Shared LSTM → GATv2 (full attention, per timestep) → Position output

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral

In [ ]:
import os
import sys

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
else:
    os.chdir('/home/adam/new4YP/4YP-main')

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from empyrical import (
    sharpe_ratio, sortino_ratio, max_drawdown,
    annual_return, annual_volatility, calmar_ratio,
)

import random
random.seed(40)
np.random.seed(40)

import tensorflow as tf
tf.random.set_seed(40)

print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
# Training/Test Configuration
TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023

VOL_TARGET = 0.15

# Stride: 20 = non-overlapping independent windows
TRAIN_STRIDE = 20

# Model Configuration
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

# GATv2 Hyperparameters (all changed from original)
HIDDEN_LAYER_SIZE = 32   # was 10
GAT_UNITS = 16           # was 8
ATTN_HEADS = 4           # was 2
LSTM_DROPOUT = 0.3       # was 0.5 (shared)
ATTN_DROPOUT = 0.1       # was 0.5 (shared)
LEARNING_RATE = 0.001    # was 0.0005
MAX_GRADIENT_NORM = 1.0  # was 0.01
NUM_GAT_LAYERS = 2
BATCH_SIZE = 128         # was 32

print(f"Train: {TRAIN_START}-{TEST_START}")
print(f"Test:  {TEST_START}-{TEST_END}")
print(f"\nModel: LSTM-GATv2 End-to-End (Step 1a)")
print(f"  Stride: {TRAIN_STRIDE}")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}, dropout: {LSTM_DROPOUT}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}, layers: {NUM_GAT_LAYERS}")
print(f"  Attention dropout: {ATTN_DROPOUT}")
print(f"  LR: {LEARNING_RATE}, clip norm: {MAX_GRADIENT_NORM}")
print(f"  Batch size: {BATCH_SIZE}")

## 3. Helper Functions

In [ ]:
def calc_daily_returns(df, returns_col='captured_returns'):
    """Aggregate daily returns across all tickers."""
    num_tickers = df['identifier'].nunique()
    daily_ret = df.groupby('time')[returns_col].sum() / num_tickers
    daily_ret.index = pd.to_datetime(daily_ret.index)
    return daily_ret.sort_index()


def calc_vol_scaled_returns(daily_returns, target_vol=0.15):
    """Scale returns to target annualized volatility."""
    current_vol = daily_returns.std() * np.sqrt(252)
    if current_vol > 0:
        return daily_returns * (target_vol / current_vol)
    return daily_returns


def calc_metrics(daily_returns, name="Strategy"):
    """Calculate all performance metrics."""
    return {
        'Strategy': name,
        'E[Ret.]': annual_return(daily_returns),
        'Vol.': annual_volatility(daily_returns),
        'Sharpe': sharpe_ratio(daily_returns),
        'Sortino': sortino_ratio(daily_returns),
        'Max DD': -max_drawdown(daily_returns),
        'Calmar': calmar_ratio(daily_returns),
        'Hit Rate': (daily_returns > 0).mean(),
        'Avg P/L': daily_returns[daily_returns > 0].mean() / abs(daily_returns[daily_returns < 0].mean()) if (daily_returns < 0).any() else np.nan,
    }


def calc_metrics_vol_normalized(daily_returns, name="Strategy", target_vol=0.15):
    """Calculate metrics with volatility-normalized returns."""
    scaled = calc_vol_scaled_returns(daily_returns, target_vol)
    return calc_metrics(scaled, name + " (Vol-Norm)"), scaled


def display_metrics(metrics_dict):
    """Display metrics in a formatted table."""
    df = pd.DataFrame([metrics_dict]).set_index('Strategy')
    for col in ['E[Ret.]', 'Vol.', 'Max DD', 'Hit Rate']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f"{x:.2%}")
    for col in ['Sharpe', 'Sortino', 'Calmar', 'Avg P/L']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f"{x:.3f}")
    display(df)
    return df


def calc_yearly_sharpes(daily_returns):
    """Calculate Sharpe ratio by year."""
    yearly = {}
    for year in sorted(daily_returns.index.year.unique()):
        yr_ret = daily_returns[daily_returns.index.year == year]
        yearly[year] = sharpe_ratio(yr_ret)
    return yearly


def plot_results(daily_returns_dict, title="Strategy Comparison"):
    """Plot cumulative returns, drawdown, and rolling Sharpe."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = plt.cm.tab10(np.linspace(0, 1, len(daily_returns_dict)))

    ax1 = axes[0, 0]
    for (name, returns), color in zip(daily_returns_dict.items(), colors):
        cum_ret = (1 + returns).cumprod() - 1
        ax1.plot(cum_ret.index, cum_ret.values, label=name, linewidth=1.5, color=color)
    ax1.set_title('Cumulative Returns')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Cumulative Return')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    for (name, returns), color in zip(daily_returns_dict.items(), colors):
        cum = (1 + returns).cumprod()
        running_max = cum.cummax()
        drawdown = (cum - running_max) / running_max
        ax2.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, label=name, color=color)
    ax2.set_title('Drawdown')
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Drawdown')
    ax2.legend(loc='lower left', fontsize=8)
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    for (name, returns), color in zip(daily_returns_dict.items(), colors):
        rolling_sharpe = returns.rolling(252).mean() / returns.rolling(252).std() * np.sqrt(252)
        ax3.plot(rolling_sharpe.index, rolling_sharpe.values, label=name, linewidth=1, color=color)
    ax3.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
    ax3.set_title('Rolling 252-Day Sharpe Ratio')
    ax3.set_xlabel('Date')
    ax3.set_ylabel('Sharpe Ratio')
    ax3.legend(loc='upper left', fontsize=8)
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    yearly_data = {name: calc_yearly_sharpes(returns) for name, returns in daily_returns_dict.items()}
    yearly_df = pd.DataFrame(yearly_data)
    yearly_df.plot(kind='bar', ax=ax4, width=0.8)
    ax4.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
    ax4.set_title('Yearly Sharpe Ratios')
    ax4.set_xlabel('Year')
    ax4.set_ylabel('Sharpe Ratio')
    ax4.legend(loc='upper right', fontsize=8)
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(True, alpha=0.3, axis='y')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Data Loading

In [ ]:
features_path = "data/straddle_features/features.csv"
df = pd.read_csv(features_path)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded {len(df)} rows")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Tickers: {df['ticker'].nunique()}")

In [ ]:
from gml.graph_model_inputs import GraphModelFeatures

features = GraphModelFeatures(
    df=df,
    total_time_steps=TOTAL_TIME_STEPS,
    start_boundary=TRAIN_START,
    test_boundary=TEST_START,
    test_end=TEST_END,
    train_valid_ratio=TRAIN_VALID_RATIO,
    split_tickers_individually=True,
    time_features=False,
    train_valid_sliding=True,
)

print("Feature generator created (sliding windows enabled).")

In [ ]:
# Apply stride for independent non-overlapping windows
train_data = {k: v[::TRAIN_STRIDE] for k, v in features.train.items()}
valid_data = {k: v[::TRAIN_STRIDE] for k, v in features.valid.items()}
test_data = features.test_sliding  # Test always uses stride=1

print(f"Stride: {TRAIN_STRIDE}")
print(f"\nTraining data:")
print(f"  inputs: {train_data['inputs'].shape}")
print(f"  outputs: {train_data['outputs'].shape}")
print(f"\nValidation data:")
print(f"  inputs: {valid_data['inputs'].shape}")
print(f"\nTest data:")
print(f"  inputs: {test_data['inputs'].shape}")

## 5. Model Definition

In [ ]:
from gml.graph_attention_v2 import build_lstm_gat_e2e_v2

num_tickers = train_data['inputs'].shape[1]
time_steps = train_data['inputs'].shape[2]
input_size = train_data['inputs'].shape[3]

print(f"Building LSTM-GATv2 E2E model (Step 1a):")
print(f"  num_tickers: {num_tickers}")
print(f"  time_steps: {time_steps}")
print(f"  input_size: {input_size}")

model = build_lstm_gat_e2e_v2(
    num_tickers=num_tickers,
    time_steps=time_steps,
    input_size=input_size,
    hidden_layer_size=HIDDEN_LAYER_SIZE,
    gat_units=GAT_UNITS,
    attn_heads=ATTN_HEADS,
    lstm_dropout=LSTM_DROPOUT,
    attn_dropout=ATTN_DROPOUT,
    learning_rate=LEARNING_RATE,
    max_gradient_norm=MAX_GRADIENT_NORM,
    num_gat_layers=NUM_GAT_LAYERS,
)

print(f"\nTotal parameters: {model.count_params():,}")

## 6. Training

In [ ]:
X_train = train_data['inputs']
y_train = train_data['outputs']
w_train = train_data['active_entries']

X_valid = valid_data['inputs']
y_valid = valid_data['outputs']
w_valid = valid_data['active_entries']

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_valid.shape[0]}")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

print("=" * 60)
print("Training LSTM-GATv2 E2E (Step 1a)")
print(f"  Stride: {TRAIN_STRIDE} ({X_train.shape[0]} training samples)")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}, GAT: {GAT_UNITS}x{ATTN_HEADS}")
print(f"  Dropout: LSTM={LSTM_DROPOUT}, Attn={ATTN_DROPOUT}")
print(f"  LR: {LEARNING_RATE}, clip: {MAX_GRADIENT_NORM}, batch: {BATCH_SIZE}")
print("=" * 60)

history = model.fit(
    X_train, y_train,
    sample_weight=w_train,
    validation_data=(X_valid, y_valid, w_valid),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Negative Sharpe)')
plt.title('LSTM-GATv2 E2E (Step 1a) Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Epochs trained: {len(history.history['loss'])}")
print(f"Best val loss: {min(history.history['val_loss']):.4f}")

## 7. Evaluation

In [ ]:
X_test = test_data['inputs']
predictions = model.predict(X_test)

print(f"Predictions shape: {predictions.shape}")
print(f"Test outputs shape: {test_data['outputs'].shape}")

In [ ]:
# Extract last timestep only (no double-counting)
positions = predictions[:, :, -1, 0].reshape(-1)
returns = test_data['outputs'][:, :, -1, 0].reshape(-1)
captured_returns = positions * returns

dates = test_data['date'][:, :, -1, 0].reshape(-1)
identifiers = test_data['identifier'][:, :, -1, 0].reshape(-1)

results_df = pd.DataFrame({
    'time': dates, 'identifier': identifiers,
    'position': positions, 'returns': returns,
    'captured_returns': captured_returns,
})

results_df['time'] = pd.to_datetime(results_df['time'])
results_df = results_df[results_df['identifier'] != '0']

print(f"Results: {len(results_df)} rows")
results_df.head()

In [ ]:
daily_returns = calc_daily_returns(results_df)

print("\n" + "=" * 60)
print("LSTM-GATv2 E2E (Step 1a) Results (Raw)")
print("=" * 60)

metrics_raw = calc_metrics(daily_returns, "LSTM-GATv2 E2E")
display_metrics(metrics_raw)

print(f"\nVolatility-Normalized (Target: {VOL_TARGET:.0%})")
metrics_norm, scaled_returns = calc_metrics_vol_normalized(daily_returns, "LSTM-GATv2 E2E", VOL_TARGET)
display_metrics(metrics_norm)

print("\nYearly Sharpe Ratios:")
yearly_sharpes = calc_yearly_sharpes(daily_returns)
for year, sharpe_val in yearly_sharpes.items():
    print(f"  {year}: {sharpe_val:.4f}")

In [ ]:
all_daily_returns = {'LSTM-GATv2 E2E (Step 1a)': daily_returns}
plot_results(all_daily_returns, f"LSTM-GATv2 E2E Step 1a ({TEST_START}-{TEST_END})")

## 8. Attention Analysis

In [ ]:
from gml.graph_attention_v2 import extract_attention_weights_v2

# Extract attention from a subset of test windows
n_samples = min(50, X_test.shape[0])
sample_indices = np.linspace(0, X_test.shape[0] - 1, n_samples, dtype=int)
X_sample = X_test[sample_indices]

attn_weights = extract_attention_weights_v2(model, X_sample, gat_layer_name="gat_v2_0")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"  (samples, heads, nodes, nodes)")

# Average across heads
attn_avg = attn_weights.mean(axis=1)  # (samples, nodes, nodes)

# Check attention entropy (should be < log(88) = 4.48 for non-degenerate)
entropy = -np.sum(attn_avg * np.log(attn_avg + 1e-9), axis=-1)  # (samples, nodes)
mean_entropy = entropy.mean()
max_entropy = np.log(attn_avg.shape[-1])
print(f"\nAttention entropy: {mean_entropy:.3f} (uniform = {max_entropy:.3f})")
print(f"Entropy ratio: {mean_entropy / max_entropy:.3f} (lower = more focused)")

In [ ]:
# Plot average attention heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Overall average attention
avg_attn = attn_avg.mean(axis=0)  # (nodes, nodes)
sns.heatmap(avg_attn, ax=axes[0], cmap='viridis', vmin=0)
axes[0].set_title('Average Attention (all samples, all heads)')
axes[0].set_xlabel('Target Stock')
axes[0].set_ylabel('Source Stock')

# Attention entropy distribution
axes[1].hist(entropy.flatten(), bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=max_entropy, color='red', linestyle='--', label=f'Uniform = {max_entropy:.2f}')
axes[1].axvline(x=mean_entropy, color='blue', linestyle='--', label=f'Mean = {mean_entropy:.2f}')
axes[1].set_xlabel('Entropy (nats)')
axes[1].set_ylabel('Count')
axes[1].set_title('Attention Entropy Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Position Analysis

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(results_df['position'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Position')
plt.ylabel('Frequency')
plt.title('Position Distribution')
plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Save Results

In [ ]:
results_dir = f"results/lstm_gat_e2e_v2_u{GAT_UNITS}_h{ATTN_HEADS}/{TEST_START}-{TEST_END}"
os.makedirs(results_dir, exist_ok=True)

results_df.to_csv(os.path.join(results_dir, "captured_returns_sw.csv"), index=False)
pd.DataFrame([metrics_raw]).to_csv(os.path.join(results_dir, "metrics_raw.csv"), index=False)
pd.DataFrame([metrics_norm]).to_csv(os.path.join(results_dir, "metrics_vol_normalized.csv"), index=False)
pd.DataFrame(yearly_sharpes.items(), columns=['Year', 'Sharpe']).to_csv(os.path.join(results_dir, "yearly_sharpes.csv"), index=False)

print(f"Results saved to: {results_dir}")

## 11. Summary

In [ ]:
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"\nModel: LSTM-GATv2 End-to-End (Step 1a)")
print(f"  GATv2 dynamic attention (truly pairwise)")
print(f"  No LayerNorm, split dropout, larger dims")
print(f"\nHyperparameters:")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}, dropout: {LSTM_DROPOUT}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}, layers: {NUM_GAT_LAYERS}")
print(f"  Attn dropout: {ATTN_DROPOUT}")
print(f"  LR: {LEARNING_RATE}, clip norm: {MAX_GRADIENT_NORM}")
print(f"  Batch size: {BATCH_SIZE}, stride: {TRAIN_STRIDE}")
print(f"\nTraining: {TRAIN_START}-{TEST_START}")
print(f"Test:     {TEST_START}-{TEST_END}")
print(f"\nPerformance (Raw):")
print(f"  Sharpe: {metrics_raw['Sharpe']:.3f}")
print(f"  Return: {metrics_raw['E[Ret.]']:.2%}")
print(f"  Vol:    {metrics_raw['Vol.']:.2%}")
print(f"  Sortino: {metrics_raw['Sortino']:.3f}")
print(f"  Max DD: {metrics_raw['Max DD']:.2%}")
print(f"\nAttention entropy: {mean_entropy:.3f} / {max_entropy:.3f} (ratio: {mean_entropy/max_entropy:.3f})")